# Module 15 · The discrete Hartley transform and the sine and cosine family

**Book source:** Bracewell, chapter 12, pp. 293–326

Companion notebook of the lecture with the same module number. Run the cells in order; each cell is followed by a **📤 Real output** note that explains the numbers.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (7.5, 3.3), "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 110})


def amplitude_spectrum(x):
    """One-sided amplitude spectrum: a sinusoid of amplitude A gives a line of height A, DC gives exactly its value"""
    a = 2*np.abs(np.fft.rfft(x))/len(x)
    a[0] /= 2
    return a


def report(key, value, fmt=".4g"):
    """Print a named result without trailing zeros (the website reads the lines that start with ▸)."""
    s = f"{value:{fmt}}"
    if "." in s and "e" not in s:
        s = s.rstrip("0").rstrip(".")
    print(f"▸ {key} = {s}")


## 1. The integral Hartley transform
🎯 **What question does this method answer?** Does the example $e^{-t}H(t)$ have $\psi=(1+\omega)/(1+\omega^2)$ and $\psi=\mathrm{Re}\,S-\mathrm{Im}\,S$, and does applying the Hartley formula twice return the function?

In [2]:
from scipy import integrate
from scipy.linalg import toeplitz
trap = getattr(np, "trapezoid", None) or np.trapz
rg = np.random.default_rng(15)
def hart(w):   # ∫ e^{-t} cas(ωt) dt, t>0
    c = integrate.quad(lambda t: np.exp(-t), 0, np.inf, weight="cos", wvar=w)[0] if w != 0 else 1.0
    s = integrate.quad(lambda t: np.exp(-t), 0, np.inf, weight="sin", wvar=w)[0] if w != 0 else 0.0
    return c + s
for w in (0.0, 1.0, 2.0, -1.0, -2.0, 0.5):
    assert abs(hart(w) - (1 + w)/(1 + w*w)) < 1e-9
    Sr = 1/(1 + w*w); Si = -w/(1 + w*w)
    assert abs((Sr - Si) - hart(w)) < 1e-9
report("ct_w0", hart(0.0), ".4f"); report("ct_w1", hart(1.0), ".4f"); report("ct_w2", hart(2.0), ".4f"); report("ct_wm1", round(hart(-1.0), 10) + 0.0, ".4f")
report("ft_re1", 0.5, ".1f"); report("ft_im1", -0.5, ".1f")
# 45° rotation
Rr, Ii = 0.5, -0.5; Rp = (np.exp(1j*np.pi/4)*(Rr + 1j*Ii)).real
assert abs(hart(1.0)/Rp - np.sqrt(2)) < 1e-12
report("rot_r", Rp, ".4f"); report("rot_ratio", hart(1.0)/Rp, ".4f")
# reciprocity: off-centre Gaussian
Vt = lambda t: np.exp(-np.pi*(t - 0.5)**2)
tg = np.linspace(-9, 9, 36001); sg = np.linspace(-9, 9, 36001)
def hart_num(g, x, y):   # ∫ g(x) cas(2π s x) dx at each y
    return np.array([trap(g*(np.cos(2*np.pi*yy*x) + np.sin(2*np.pi*yy*x)), x) for yy in y])
s_pts = np.linspace(-6, 6, 2401)
psi_num = hart_num(Vt(tg), tg, s_pts); psi_cf = np.exp(-np.pi*s_pts**2)*(np.cos(np.pi*s_pts) + np.sin(np.pi*s_pts))
assert np.max(np.abs(psi_num - psi_cf)) < 1e-9
t0 = 0.3
rec = trap(psi_cf*(np.cos(2*np.pi*s_pts*t0) + np.sin(2*np.pi*s_pts*t0)), s_pts)
assert abs(rec - Vt(t0)) < 1e-9
report("rec_v", Vt(t0), ".4f"); report("rec_dev", max(abs(rec - Vt(t0)), 1e-16), ".0e")

▸ ct_w0 = 1
▸ ct_w1 = 1
▸ ct_w2 = 0.6
▸ ct_wm1 = 0
▸ ft_re1 = 0.5
▸ ft_im1 = -0.5
▸ rot_r = 0.7071
▸ rot_ratio = 1.4142


▸ rec_v = 0.8819
▸ rec_dev = 1e-16


#### 📤 Real output
$\psi(0)$ = 1, $\psi(1)$ = 1, $\psi(2)$ = 0.6, $\psi(-1)$ = 0; 45 degree rotation: 0.7071, ratio 1.4142; reciprocity: 0.8819, deviation 1e-16.

## 2. The DHT: definition, examples, counting reals
🎯 **What question does this method answer?** Does our DHT match Bracewell's short sequences, does the inverse have the same formula, and do $F=E-iO$ and $H=\mathrm{Re}F-\mathrm{Im}F$ hold?

In [3]:
def cas(x): return np.cos(x) + np.sin(x)
def dht(f):
    f = np.asarray(f, float); N = len(f); t = np.arange(N)
    return np.array([np.sum(f*cas(2*np.pi*v*t/N)) for v in range(N)])/N
def idht(H):
    H = np.asarray(H, float); N = len(H); v = np.arange(N)
    return np.array([np.sum(H*cas(2*np.pi*v*t/N)) for t in range(N)])
assert np.allclose(dht([1, 2]), [1.5, -0.5]); report("n2_a", 1.5, ".1f"); report("n2_b", -0.5, ".1f")
h4 = dht([1, 2, 3, 4]); assert np.allclose(h4, [2.5, -1, -0.5, 0]); report("h4_1", h4[1], ".1f")
f8 = np.arange(1, 9.); h8 = dht(f8)
assert np.allclose(h8, [4.5, -1.7071, -1, -0.7071, -0.5, -0.2929, 0, 0.7071], atol=1e-4)
report("h8_1", h8[1], ".4f"); report("h8_3", h8[3], ".4f"); report("h8_5", h8[5], ".4f"); report("h8_7", h8[7], ".4f"); report("h8_zero", round(h8[6], 12) + 0.0, ".0f"); report("z8", int(np.argmin(np.abs(h8[1:])) + 1), "d")
assert abs(h8.sum() - f8[0]) < 1e-12 and abs(f8.sum() - 8*h8[0]) < 1e-12
report("sumH", h8.sum(), ".0f"); report("sumf", f8.sum(), ".0f")
Nn = 16; tt = np.arange(Nn)
Ogm = np.array([[np.sum(cas(2*np.pi*tt*a/Nn)*cas(2*np.pi*tt*b/Nn)) for b in range(Nn)] for a in range(Nn)])
assert np.allclose(Ogm, Nn*np.eye(Nn)); report("og_dev", max(np.max(np.abs(Ogm - Nn*np.eye(Nn))), 1e-16), ".0e")
xr = rg.standard_normal(16); assert np.allclose(idht(dht(xr)), xr) and np.allclose(dht(dht(xr))*16, xr)
report("inv15", max(np.max(np.abs(idht(dht(xr)) - xr)), 1e-16), ".0e")
# normalised involution
N8 = 8; Cm = cas(2*np.pi*np.outer(np.arange(N8), np.arange(N8))/N8)/np.sqrt(N8)
assert np.allclose(Cm, Cm.T) and np.allclose(Cm @ Cm, np.eye(N8)) and np.all(np.abs(np.abs(np.linalg.eigvalsh(Cm)) - 1) < 1e-12)
report("sym_dev", max(np.max(np.abs(Cm @ Cm - np.eye(N8))), 1e-16), ".0e")
# F = E − iO, H = Re F − Im F
xf = rg.standard_normal(16); Hf = dht(xf); nu = np.arange(16); Hr = Hf[(-nu) % 16]
E = 0.5*(Hf + Hr); O = 0.5*(Hf - Hr); Ff = np.fft.fft(xf)/16
assert np.allclose(E - 1j*O, Ff) and np.allclose(Ff.real - Ff.imag, Hf)
report("fdev", max(np.max(np.abs(E - 1j*O - Ff)), 1e-16), ".0e"); report("hdev", max(np.max(np.abs(Ff.real - Ff.imag - Hf)), 1e-16), ".0e")
# truncated exponential N = 16
fe = np.exp(-np.arange(16)/2); fe[0] = 0.5; He = dht(fe)
psi = lambda s: (0.5 + 2*np.pi*s)/(0.25 + 4*np.pi**2*s**2)
assert abs(He[1] - psi(1/16)/16) < 0.02*abs(He[1])
report("ex_H0", He[0], ".4f"); report("ex_H1", He[1], ".4f"); report("ex_c1", psi(1/16)/16, ".4f")
# binomial
bn = np.array([20, 15, 6, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 6, 15.]); Hb = dht(bn)
assert np.allclose(Hb, np.fft.fft(bn).real/16) and np.allclose(Hb, Hb[(-np.arange(16)) % 16])
assert abs(Hb[4] - 0.5) < 1e-12 and abs(Hb[1] - 3.56) < 0.005 and abs(Hb[2] - 2.49) < 0.005
report("bn_H1", Hb[1], ".2f"); report("bn_H2", Hb[2], ".2f"); report("bn_H4", Hb[4], ".1f")
# general kernel √2 sin(θ+α)
al = 0.6; xg = rg.standard_normal(16); th = 2*np.pi*np.outer(np.arange(16), np.arange(16))/16
G = (np.sqrt(2)*np.sin(th + al)) @ xg
rec_g = (1/16)*(1/np.sqrt(2))*((np.cos(th.T)/np.sin(al) + np.sin(th.T)/np.cos(al)) @ G)
assert np.allclose(rec_g, xg)
report("gk_dev", max(np.max(np.abs(rec_g - xg)), 1e-16), ".0e")

▸ n2_a = 1.5
▸ n2_b = -0.5
▸ h4_1 = -1
▸ h8_1 = -1.7071
▸ h8_3 = -0.7071
▸ h8_5 = -0.2929
▸ h8_7 = 0.7071
▸ h8_zero = 0
▸ z8 = 6
▸ sumH = 1
▸ sumf = 36
▸ og_dev = 5e-14
▸ inv15 = 6e-15
▸ sym_dev = 2e-15
▸ fdev = 8e-16
▸ hdev = 1e-15
▸ ex_H0 = 0.1275
▸ ex_H1 = 0.1386
▸ ex_c1 = 0.138
▸ bn_H1 = 3.56
▸ bn_H2 = 2.49
▸ bn_H4 = 0.5
▸ gk_dev = 6e-15


#### 📤 Real output
$\{1,2\}\to$ 1.5; $\{1..4\}$: $H(1)$ = -1; $\{1..8\}$: -1.7071, -0.7071, -0.2929, 0.7071, zero at $\nu$ = 6; sums 1 and 36; orthogonality 5e-14; inverse 6e-15; involution 2e-15; $F=E-iO$ deviates 8e-16; truncated exponential 0.1275, 0.1386, 0.138; binomial 3.56, 2.49, 0.5; general kernel 6e-15.

## 3. Theorems, convolution, two dimensions, power spectrum
🎯 **What question does this method answer?** Do the shift, reversal, difference, Parseval and convolution theorems hold numerically, do the 2D DHT and cas-cas behave as stated, and does the power spectrum read straight from $H$?

In [4]:
N = 16; nu = np.arange(N); xr = rg.standard_normal(N); Hx = dht(xr)
# Parseval, reversal
assert abs(np.sum(f8**2) - 8*np.sum(h8**2)) < 1e-9 and abs(np.sum(xr**2) - N*np.sum(Hx**2)) < 1e-9
report("qc_l", np.sum(f8**2), ".0f"); report("qc_r", 8*np.sum(h8**2), ".0f")
rv = dht(xr[(-nu) % N]); assert np.allclose(rv, Hx[(-nu) % N]); report("rv_dev", max(np.max(np.abs(rv - Hx[(-nu) % N])), 1e-16), ".0e")
Fq = np.fft.fft(xr)/N; assert abs(np.sum(np.abs(Fq)**2) - 0.5*np.sum(Hx**2 + Hx[(-nu) % N]**2)) < 1e-12
# shift, difference
a = 3; c = np.cos(2*np.pi*nu*a/N); s = np.sin(2*np.pi*nu*a/N)
lhs = dht(np.roll(xr, -a)); right = Hx*c - Hx[(-nu) % N]*s; wrong = Hx*c + Hx[(-nu) % N]*s
assert np.allclose(lhs, right) and not np.allclose(lhs, wrong)
report("sh_dev", max(np.max(np.abs(lhs - right)), 1e-16), ".0e"); report("sh_wrong", np.max(np.abs(lhs - wrong)), ".1f")
c1 = np.cos(2*np.pi*nu/N); s1 = np.sin(2*np.pi*nu/N)
dl = dht(np.roll(xr, -1) - xr); dr = (c1 - 1)*Hx - s1*Hx[(-nu) % N]
assert np.allclose(dl, dr); report("df_dev", max(np.max(np.abs(dl - dr)), 1e-16), ".0e")
# convolution
def cconv(a_, b_):
    n = len(a_); return np.array([sum(a_[k]*b_[(t - k) % n] for k in range(n)) for t in range(n)])
f1 = rg.standard_normal(N); f2 = rg.standard_normal(N); H1 = dht(f1); H2 = dht(f2); Hm2 = H2[(-nu) % N]
H2e = 0.5*(H2 + Hm2); H2o = 0.5*(H2 - Hm2)
Hc = dht(cconv(f1, f2)); Hth = N*(H1*H2e + H1[(-nu) % N]*H2o)
assert np.allclose(Hc, Hth); report("cv_dev", max(np.max(np.abs(Hc - Hth)), 1e-16), ".0e")
report("mul_dft", 4, "d"); report("mul_dht", 2, "d")
# the procedure with an even filter and zero padding
Np = 16; d = np.zeros(Np); d[:4] = [1, 2, 3, 4]
kern = np.zeros(Np); kern[0] = 0.5; kern[1] = 0.25; kern[-1] = 0.25
assert np.allclose(kern, kern[(-np.arange(Np)) % Np])
conv_h = idht(Np*dht(d)*dht(kern))
assert np.allclose(conv_h, cconv(d, kern))
direct = np.convolve([1, 2, 3, 4.], [0.25, 0.5, 0.25])
res = cconv(d, kern); shifted = np.roll(res, 1)[:6]
assert np.allclose(shifted, direct)
report("cp_dev", max(np.max(np.abs(conv_h - res)), 1e-16), ".0e"); report("cp_max", direct.max(), ".2f")
# two dimensions
A = rg.standard_normal((6, 5)); M_, N_ = A.shape
t1 = np.arange(M_)[:, None]; t2 = np.arange(N_)[None, :]
def dht2(X):
    out = np.zeros(X.shape); t1 = np.arange(X.shape[0])[:, None]; t2 = np.arange(X.shape[1])[None, :]
    for u in range(X.shape[0]):
        for v in range(X.shape[1]):
            out[u, v] = np.sum(X*cas(2*np.pi*(u*t1/X.shape[0] + v*t2/X.shape[1])))/X.size
    return out
H2d = dht2(A); F2 = np.fft.fft2(A)/A.size
assert np.allclose(H2d, F2.real - F2.imag) and np.allclose(dht2(H2d)*A.size, A)
report("d2_dev", max(np.max(np.abs(H2d - (F2.real - F2.imag))), 1e-16), ".0e"); report("d2_inv", max(np.max(np.abs(dht2(H2d)*A.size - A)), 1e-16), ".0e")
# cas-cas
B = rg.standard_normal((8, 8)); ax = np.arange(8)
Ccc = cas(2*np.pi*np.outer(ax, ax)/8)/np.sqrt(8)
Hcc = Ccc @ B @ Ccc; assert np.allclose(Ccc @ Hcc @ Ccc, B)
Hst = dht2(B)*8
report("cc_rec", max(np.max(np.abs(Ccc @ Hcc @ Ccc - B)), 1e-16), ".0e"); report("cc_diff", np.max(np.abs(Hcc - Hst)), ".2f")
# power spectrum, Problem 4
f87 = np.arange(8, 0, -1.); Hs = dht(f87); n8 = np.arange(8); Hr8 = Hs[(-n8) % 8]
assert np.allclose(Hs, [4.5, 1.7071, 1, 0.7071, 0.5, 0.2929, 0, -0.7071], atol=1e-4)
E1 = 0.5*(Hs[1] + Hr8[1]); O1 = 0.5*(Hs[1] - Hr8[1]); F87 = np.fft.fft(f87)/8
assert abs(E1 - F87[1].real) < 1e-12 and abs(O1 + F87[1].imag) < 1e-12
P1 = 0.5*(Hs[1]**2 + Hr8[1]**2); assert abs(P1 - abs(F87[1])**2) < 1e-12
report("p4_E1", E1, ".1f"); report("p4_O1", O1, ".4f"); report("p4_P1", P1, ".4f"); report("p4_dev", max(abs(P1 - abs(F87[1])**2), 1e-16), ".0e")
Pdir = 0.5*(Hx**2 + Hx[(-nu) % N]**2); assert np.allclose(Pdir, np.abs(Fq)**2)
report("ps_dev", max(np.max(np.abs(Pdir - np.abs(Fq)**2)), 1e-16), ".0e")
# exercises
def zero_idx(P):
    Nq = 2**P; hh = dht(np.arange(1, Nq + 1.)); z = np.where(np.abs(hh) < 1e-9)[0]; assert len(z) == 1; return int(z[0])
assert zero_idx(2) == 3 and zero_idx(3) == 6 and zero_idx(4) == 12 and zero_idx(6) == 48
report("z16", zero_idx(4), "d"); report("z64", zero_idx(6), "d")
rp = dht(np.tile([1, 2, 3, 4.], 4)); nz = np.where(np.abs(rp) > 1e-9)[0]; assert list(nz) == [0, 4, 8]
report("rp_0", rp[0], ".1f"); report("rp_4", rp[4], ".1f"); report("rp_8", rp[8], ".1f")
al8 = dht([-2, 2]*4); assert list(np.where(np.abs(al8) > 1e-9)[0]) == [4]; report("alt", al8[4], ".1f")

▸ qc_l = 204
▸ qc_r = 204
▸ rv_dev = 1e-15
▸ sh_dev = 2e-15
▸ sh_wrong = 0.7
▸ df_dev = 1e-15
▸ cv_dev = 1e-14
▸ mul_dft = 4
▸ mul_dht = 2
▸ cp_dev = 4e-15
▸ cp_max = 3
▸ d2_dev = 6e-16
▸ d2_inv = 4e-15
▸ cc_rec = 6e-15
▸ cc_diff = 1.52
▸ p4_E1 = 0.5
▸ p4_O1 = 1.2071
▸ p4_P1 = 1.7071
▸ p4_dev = 2e-15
▸ ps_dev = 3e-16
▸ z16 = 12
▸ z64 = 48
▸ rp_0 = 2.5
▸ rp_4 = -1
▸ rp_8 = -0.5
▸ alt = -2


#### 📤 Real output
Parseval 204 and 204; shift deviates 2e-15, wrong sign 0.7; difference 1e-15; convolution 1e-14, even filter 4e-15 (largest 3); 2D 6e-16, 4e-15; cas-cas 6e-15, difference 1.52; Problem 4: 0.5, 1.2071, 1.7071; zero indices 12, 48; repetition 2.5, -1, -0.5; Nyquist -2.

## 4. The fast Hartley transform
🎯 **What question does this method answer?** Does the decomposition formula hold, does the hand example $\{1,\ldots,8\}$ reproduce Bracewell's numbers, how many indices does the permutation fix, and how do the multiplication counts compare with the FFT?

In [5]:
# decomposition
def fht_rec(f, cnt):
    Nq = len(f)
    if Nq == 1: return f.copy()
    if Nq == 2: return np.array([f[0] + f[1], f[0] - f[1]])
    Ha = fht_rec(f[0::2], cnt); Hb2 = fht_rec(f[1::2], cnt); m = Nq//2; out = np.zeros(Nq)
    for v in range(Nq):
        cc_ = np.cos(2*np.pi*v/Nq); ss_ = np.sin(2*np.pi*v/Nq)
        for w_ in (cc_, ss_):
            if min(abs(w_), abs(w_ - 1), abs(w_ + 1)) > 1e-12: cnt["gen"] += 1
        out[v] = Ha[v % m] + Hb2[v % m]*cc_ + Hb2[(-v) % m]*ss_
    return out
xd = rg.standard_normal(16)
Hd1 = dht(xd[0::2]); Hd2 = dht(xd[1::2]); vv = np.arange(16); mm = 8
dec = 0.5*(Hd1[vv % mm] + Hd2[vv % mm]*np.cos(2*np.pi*vv/16) + Hd2[(-vv) % mm]*np.sin(2*np.pi*vv/16))
assert np.allclose(dec, dht(xd)); report("dc_dev", max(np.max(np.abs(dec - dht(xd))), 1e-16), ".0e")
# hand example
f = np.arange(1, 9.)
perm = [0, 4, 2, 6, 1, 5, 3, 7]; fp = f[perm]; assert list(fp) == [1, 5, 3, 7, 2, 6, 4, 8]
f1 = np.concatenate([[fp[i] + fp[i+1], fp[i] - fp[i+1]] for i in range(0, 8, 2)])
assert list(f1) == [6, -4, 10, -4, 8, -4, 12, -4]
cnt8 = dict(gen=0); f3 = fht_rec(f, cnt8); assert np.allclose(f3, 8*dht(f))
report("st1_a", f1[0], ".0f"); report("st1_b", f1[1], ".0f"); report("f3_1", f3[1], ".4f")
# permutation
def brv(n, bits): return int(format(n, "0%db" % bits)[::-1], 2)
def fixed(Pw): return sum(1 for i in range(2**Pw) if brv(i, Pw) == i)
assert fixed(3) == 4 and fixed(10) == 32 and fixed(2) == 2 and fixed(4) == 4
sw = [(i, brv(i, 3)) for i in range(8) if brv(i, 3) > i]; assert sw == [(1, 4), (3, 6)]
report("fx8", fixed(3), "d"); report("fx1024", fixed(10), "d"); report("sw8", len(sw), "d")
# counting
cN = dict(gen=0); x1k = rg.standard_normal(1024); assert np.allclose(fht_rec(x1k, cN), 1024*dht(x1k))
def fft_gen(x):
    cntf = dict(gen=0)
    def rec(x_):
        n = len(x_)
        if n == 1: return x_.copy()
        Ee = rec(x_[0::2]); Oo = rec(x_[1::2]); Wk = np.exp(-2j*np.pi*np.arange(n//2)/n)
        for w_ in Wk:
            if min(abs(w_ - 1), abs(w_ + 1), abs(w_ - 1j), abs(w_ + 1j)) > 1e-12: cntf["gen"] += 1
        return np.concatenate([Ee + Wk*Oo, Ee - Wk*Oo])
    rec(x.astype(complex)); return cntf["gen"]
ffg = fft_gen(x1k)
assert cN["gen"] == 4*ffg
report("fh_mul", cN["gen"], "d"); report("ff_gen", ffg, "d"); report("ff_real", 4*ffg, "d")
# matrices
Cm16 = cas(2*np.pi*np.outer(np.arange(16), np.arange(16))/16)
assert np.count_nonzero(np.abs(Cm16) > 1e-12) == Cm16.size - np.count_nonzero(np.abs(Cm16) <= 1e-12)
report("mt_nz", np.count_nonzero(np.abs(Cm16) > 1e-12), "d")

▸ dc_dev = 4e-15
▸ st1_a = 6
▸ st1_b = -4
▸ f3_1 = -13.6569
▸ fx8 = 4
▸ fx1024 = 32
▸ sw8 = 2
▸ fh_mul = 14344
▸ ff_gen = 3586
▸ ff_real = 14344
▸ mt_nz = 224


#### 📤 Real output
The decomposition deviates 4e-15; hand example 6, -4, $8H(1)$ = -13.6569; permutation: 4, 32, 2; $N=1024$: FHT 14344 real, FFT 3586 complex = 14344 real; the $16\times16$ matrix has 224 nonzero elements.

## 5. DST, DCT, the string and data compression
🎯 **What question does this method answer?** Are the sine and cosine transforms invertible, does the sine series of a plucked string converge, and how much energy does DCT-2 keep compared with the KLT for Markov data?

In [6]:
# DFT-type cosine transform
Nc = 8; dat = np.arange(1, 9.); rev = dat[::-1]
Ct = lambda x_: np.array([np.sum(x_*np.cos(2*np.pi*v*np.arange(Nc)/Nc)) for v in range(Nc)])
assert not np.allclose(dat, rev)
# same cosine transform for the reflection τ → −τ mod N
refl = dat[(-np.arange(Nc)) % Nc]
assert np.allclose(Ct(dat), Ct(refl)) and not np.allclose(dat, refl)
report("ct_same", max(np.max(np.abs(Ct(dat) - Ct(refl))), 1e-16), ".0e"); report("ct_diff", np.max(np.abs(dat - refl)), ".0f")
# DST
N16 = 16; tau = np.arange(1, N16); nuu = np.arange(1, N16)
Sm = np.sin(np.pi*np.outer(nuu, tau)/N16); fs = rg.standard_normal(N16 - 1)
Fs = (2/N16)*Sm @ fs; back = Sm.T @ Fs
assert np.allclose(back, fs) and np.allclose(Sm.T @ (Sm @ fs), (N16/2)*fs)
report("dst_dev", max(np.max(np.abs(back - fs)), 1e-16), ".0e"); report("dst_twice", N16/2, ".0f")
# DCT-1
Nd = 16; kk = np.where((np.arange(Nd + 1) == 0) | (np.arange(Nd + 1) == Nd), 1/np.sqrt(2), 1.0)
D1 = np.sqrt(2/Nd)*kk[:, None]*np.cos(np.pi*np.outer(np.arange(Nd + 1), np.arange(Nd + 1))/Nd)*kk[None, :]
fd = rg.standard_normal(Nd + 1); assert np.allclose(D1 @ (D1 @ fd), fd)
report("dct1_dev", max(np.max(np.abs(D1 @ (D1 @ fd) - fd)), 1e-16), ".0e")
# DCT-2
Nk = 16; kx = np.arange(Nk)
C2 = np.array([[(np.sqrt(1/Nk) if v == 0 else np.sqrt(2/Nk))*np.cos(np.pi*v*(2*t + 1)/(2*Nk)) for t in kx] for v in kx])
assert np.allclose(C2 @ C2.T, np.eye(Nk))
report("dct2_orth", max(np.max(np.abs(C2 @ C2.T - np.eye(Nk))), 1e-16), ".0e"); report("dct2_inv", np.linalg.norm(C2 @ C2 - np.eye(Nk)), ".2f")
# plucked string
a_ = 0.3
y = lambda x: np.where(x < a_, x/a_, (1 - x)/(1 - a_))
bn_ = lambda n: 2*np.sin(n*np.pi*a_)/(n**2*np.pi**2*a_*(1 - a_))
for n in (1, 2, 3):
    num = 2*integrate.quad(lambda x: y(x)*np.sin(n*np.pi*x), 0, a_, limit=200)[0] + 2*integrate.quad(lambda x: y(x)*np.sin(n*np.pi*x), a_, 1, limit=200)[0]
    assert abs(num - bn_(n)) < 1e-10
ser = sum(bn_(n)*np.sin(n*np.pi*0.5) for n in range(1, 301))
assert abs(ser - y(0.5)) < 1e-4
report("st_b1", bn_(1), ".4f"); report("st_b2", bn_(2), ".4f"); report("st_y", ser, ".4f"); report("st_yt", float(y(0.5)), ".4f"); report("st_dev", abs(ser - y(0.5)), ".0e")
# second differences
Ns = 50; Asd = 2*np.eye(Ns) - np.eye(Ns, k=1) - np.eye(Ns, k=-1)
ev_ = np.linalg.eigvalsh(Asd); ev_f = np.sort(2 - 2*np.cos(np.arange(1, Ns + 1)*np.pi/(Ns + 1)))
assert np.allclose(ev_, ev_f); report("ev_sd", max(np.max(np.abs(ev_ - ev_f)), 1e-16), ".0e")
# compression
report("cmp_x4", 4, "d"); report("cmp_x32", 256//8, "d")
rho = 0.95; Rm = toeplitz(rho**np.arange(Nk)); tot = np.trace(Rm)
def top4(T_): return np.sort(np.diag(T_ @ Rm @ T_.T))[::-1][:4].sum()/tot
Hn = cas(2*np.pi*np.outer(kx, kx)/Nk)/np.sqrt(Nk)
klt = np.sort(np.linalg.eigvalsh(Rm))[::-1][:4].sum()/tot
c_dct = top4(C2); c_dht = top4(Hn); c_id = top4(np.eye(Nk))
assert c_id < c_dht < c_dct <= klt + 1e-12 and klt - c_dct < 0.001
report("cp_dct", c_dct, ".4f"); report("cp_dht", c_dht, ".4f"); report("cp_klt", klt, ".4f"); report("cp_id", c_id, ".2f")
# feel for indices
cs = dht([1, 0, -1, 0, 1, 0, -1, 0]); sn = dht([0, 1, 0, -1, 0, 1, 0, -1]); dcs = dht([3, 2, 1, 2, 3, 2, 1, 2])
assert np.allclose(cs, [0, 0, .5, 0, 0, 0, .5, 0]) and np.allclose(sn, [0, 0, .5, 0, 0, 0, -.5, 0]) and np.allclose(dcs, [2, 0, .5, 0, 0, 0, .5, 0])
report("cs_2", cs[2], ".1f"); report("cs_6", cs[6], ".1f"); report("sn_2", sn[2], ".1f"); report("sn_6", sn[6], ".1f"); report("dc_0", dcs[0], ".0f")

▸ ct_same = 2e-14
▸ ct_diff = 6
▸ dst_dev = 3e-15
▸ dst_twice = 8
▸ dct1_dev = 2e-15
▸ dct2_orth = 2e-15
▸ dct2_inv = 1.99
▸ st_b1 = 0.7807
▸ st_b2 = 0.2294
▸ st_y = 0.7143
▸ st_yt = 0.7143
▸ st_dev = 8e-08
▸ ev_sd = 2e-15
▸ cmp_x4 = 4
▸ cmp_x32 = 32
▸ cp_dct = 0.9557
▸ cp_dht = 0.9364
▸ cp_klt = 0.9559
▸ cp_id = 0.25
▸ cs_2 = 0.5
▸ cs_6 = 0.5
▸ sn_2 = 0.5
▸ sn_6 = -0.5
▸ dc_0 = 2


#### 📤 Real output
Cosine: 2e-14; DST 3e-15, factor 8; DCT-1 2e-15; DCT-2 2e-15 and 1.99; string 0.7807, 0.2294, 0.7143; compression: DCT-2 0.9557, DHT 0.9364, KLT 0.9559, no transform 0.25; cosine/sine/dc 0.5, -0.5, 2.